In [2]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

# load cleaned crashes
df = pd.read_csv("../data/processed/crashes_cleaned.csv")
print(f"Crashes loaded: {len(df):,}")

# convert to GeoDataFrame
crashes_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
    crs="EPSG:4326"
)

# load road network
roads = gpd.read_file("../data/external/nyc_roads.gpkg")
print(f"Road segments loaded: {len(roads):,}")

# reproject both to EPSG:3857 (meters) for accurate distance
crashes_gdf = crashes_gdf.to_crs("EPSG:3857")
roads = roads.to_crs("EPSG:3857")

print("Running spatial join...")
joined = gpd.sjoin_nearest(
    crashes_gdf, 
    roads[['physicalid', 'street_name', 'rw_type', 'posted_speed', 
           'number_total_lanes', 'trafdir', 'geometry']], 
    how='left', 
    max_distance=50
)
print(f"Joined: {len(joined):,} rows")
print(f"Unmatched: {joined['physicalid'].isnull().sum():,}")

# aggregate CSI per road segment
segment_csi = joined.groupby('physicalid').agg(
    total_crashes=('csi', 'count'),
    total_csi=('csi', 'sum'),
    total_injuries=('number_of_persons_injured', 'sum'),
    total_fatalities=('number_of_persons_killed', 'sum')
).reset_index()

# bin into risk classes
segment_csi['risk_class'] = pd.qcut(
    segment_csi['total_csi'],
    q=[0, 0.7, 0.9, 1.0],
    labels=['Low', 'Moderate', 'High']
)

print("\nRisk class distribution:")
print(segment_csi['risk_class'].value_counts())

segment_csi.to_csv("../data/processed/segment_risk.csv", index=False)
print("\nSaved to data/processed/segment_risk.csv")

Crashes loaded: 461,425
Road segments loaded: 122,272
Running spatial join...
Joined: 466,752 rows
Unmatched: 2,118

Risk class distribution:
risk_class
Low         44664
Moderate    12328
High         6273
Name: count, dtype: int64

Saved to data/processed/segment_risk.csv
